# 04c — Prompting evaluation: Phi-4-mini

**Project:** Lightweight domain adaptation of small LLMs for chemistry using LoRA and QLoRA (MSc FYP).

**This notebook (04c):** completes the prompting grid by running Phi-4-mini-instruct on all three datasets under all three regimes. Same evaluation harness as 04b — the only thing that changes is the model.

**Why Phi-4-mini is the third model:**
- ~3.8B params, Microsoft's instruction-tuned small model
- Trained on heavily curated data — a different recipe from SmolLM3-3B
- Lets us answer: “is it parameter count, or training data quality, that helps small LLMs prompt-classify chemistry?”

**Note on access:** Phi-4-mini-instruct is sometimes gated on Hugging Face. If `AutoModelForCausalLM.from_pretrained` returns 401, run `from huggingface_hub import login; login()` in a new cell and paste your HF token (one with read access; create at https://huggingface.co/settings/tokens). Then re-run the load cell.

**Estimated runtime on T4:** ~30–60 minutes for the full 9-cell grid. ESOL retrieval is the slow tail.

## 1. Install + imports

In [ ]:
%pip install -q transformers bitsandbytes accelerate rdkit pandas scikit-learn

In [ ]:
import os, random, gc, time
from dataclasses import dataclass
from typing import List, Optional

import numpy as np
import pandas as pd
import torch
from rdkit import Chem, RDLogger, DataStructs
from rdkit.Chem import AllChem
from sklearn.metrics import roc_auc_score, f1_score, mean_squared_error, mean_absolute_error
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

RDLogger.DisableLog("rdApp.*")
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU   :", torch.cuda.get_device_name(0))
    print("VRAM  :", f"{torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GiB")

## 2. Mount Drive + load splits

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DATA_DIR    = "/content/drive/MyDrive/chemistry-peft-fyp/data"
RESULTS_DIR = "/content/drive/MyDrive/chemistry-peft-fyp/results"
RESULTS_CSV = f"{RESULTS_DIR}/baselines.csv"
os.makedirs(RESULTS_DIR, exist_ok=True)

def load_splits(name):
    p = name.lower()
    return (
        pd.read_csv(f"{DATA_DIR}/{p}_train.csv"),
        pd.read_csv(f"{DATA_DIR}/{p}_test.csv"),
    )

splits = {name: load_splits(name) for name in ["BBBP", "BACE", "ESOL"]}
for n, (tr, te) in splits.items():
    print(f"{n:5s}: train={len(tr)}  test={len(te)}")

## 3. Re-implement the 04a/04b helpers

Self-contained so the notebook survives a fresh Colab VM. Identical to 04b.

In [ ]:
ESOL_BUCKETS = [
    ("very_low",  -np.inf, -6.0, -7.5),
    ("low",       -6.0,    -4.0, -5.0),
    ("medium",    -4.0,    -2.0, -3.0),
    ("high",      -2.0,     0.0, -1.0),
    ("very_high",  0.0,    np.inf, 1.0),
]
ESOL_LABEL_WORDS = [b[0] for b in ESOL_BUCKETS]
ESOL_MIDPOINTS   = np.array([b[3] for b in ESOL_BUCKETS], dtype=np.float32)

def esol_continuous_to_bucket(val):
    for label, lo, hi, _ in ESOL_BUCKETS:
        if lo < val <= hi:
            return label
    return ESOL_BUCKETS[0][0]

@dataclass
class DatasetConfig:
    name: str
    task: str
    question: str
    label_words: List[str]
    positive_word: Optional[str]
    primary_metric: str

DATASETS = {
    "BBBP": DatasetConfig("BBBP", "classification",
        "Does this molecule cross the blood-brain barrier?",
        ["yes", "no"], "yes", "roc_auc"),
    "BACE": DatasetConfig("BACE", "classification",
        "Does this molecule inhibit the BACE-1 enzyme?",
        ["yes", "no"], "yes", "roc_auc"),
    "ESOL": DatasetConfig("ESOL", "regression",
        "What is the aqueous solubility of this molecule?",
        ESOL_LABEL_WORDS, None, "rmse"),
}

def label_for_prompt(cfg, raw_label):
    if cfg.task == "classification":
        return cfg.positive_word if int(raw_label) == 1 else "no"
    return esol_continuous_to_bucket(float(raw_label))

def build_prompt(cfg, test_smiles, exemplars=None):
    parts = []
    if exemplars:
        for smi, lab in exemplars:
            parts.append(f"{cfg.question}\nSMILES: {smi}\nAnswer: {lab}\n")
    parts.append(f"{cfg.question}\nSMILES: {test_smiles}\nAnswer:")
    return "\n".join(parts)

def label_token_ids(tokenizer, label_words):
    out = []
    for w in label_words:
        cands = set()
        for variant in (" " + w, w):
            t = tokenizer(variant, add_special_tokens=False).input_ids
            if t:
                cands.add(t[0])
        out.append(sorted(cands))
    return out

@torch.no_grad()
def score_labels(model, tokenizer, prompt, label_words):
    enc = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(model.device)
    out = model(**enc)
    next_logits = out.logits[0, -1, :]
    log_probs = torch.log_softmax(next_logits, dim=-1)
    token_ids_per_label = label_token_ids(tokenizer, label_words)
    label_logps = []
    for ids in token_ids_per_label:
        label_logps.append(max(log_probs[i].item() for i in ids))
    logps = np.array(label_logps, dtype=np.float64)
    logps -= logps.max()
    probs = np.exp(logps); probs /= probs.sum()
    return probs

_FPGEN = AllChem.GetMorganGenerator(radius=2, fpSize=2048)
def smiles_to_bitvect(smi):
    mol = Chem.MolFromSmiles(smi)
    return None if mol is None else _FPGEN.GetFingerprint(mol)

class TanimotoIndex:
    def __init__(self, train_df):
        self.smiles = train_df["smiles"].tolist()
        self.labels = train_df["label"].tolist()
        self.fps, self.valid_idx = [], []
        for i, s in enumerate(self.smiles):
            fp = smiles_to_bitvect(s)
            if fp is not None:
                self.fps.append(fp); self.valid_idx.append(i)
    def topk(self, query_smiles, k=5):
        q = smiles_to_bitvect(query_smiles)
        if q is None: return []
        sims = DataStructs.BulkTanimotoSimilarity(q, self.fps)
        order = np.argsort(sims)[::-1][:k]
        return [(self.smiles[self.valid_idx[i]], self.labels[self.valid_idx[i]], float(sims[i])) for i in order]

def random_exemplars(cfg, train_df, k, rng):
    idx = rng.sample(range(len(train_df)), k)
    return [(train_df["smiles"].iloc[i], label_for_prompt(cfg, train_df["label"].iloc[i])) for i in idx]

def retrieval_exemplars(cfg, index, test_smiles, k):
    return [(smi, label_for_prompt(cfg, lab)) for smi, lab, _ in index.topk(test_smiles, k=k)]

print("Helpers ready.")

## 4. Build retrieval indexes

In [ ]:
indexes = {}
for n, (tr, _) in splits.items():
    t0 = time.time()
    indexes[n] = TanimotoIndex(tr)
    print(f"{n:5s}: indexed {len(indexes[n].fps)} train mols in {time.time()-t0:.1f}s")

## 5. Evaluation harness

In [ ]:
K_SHOT = 5

def evaluate(model, tokenizer, model_name, dataset_name, regime, n_train, n_test_rows):
    cfg = DATASETS[dataset_name]
    tr_df, te_df = splits[dataset_name]
    index = indexes[dataset_name]
    rng = random.Random(SEED + hash((dataset_name, regime)) % (2**32))

    probs_all, y_true = [], []

    t0 = time.time()
    for i, row in te_df.iterrows():
        smi, y = row["smiles"], row["label"]
        if regime == "zero":
            exemplars = None
        elif regime == "few":
            exemplars = random_exemplars(cfg, tr_df, K_SHOT, rng)
        else:
            exemplars = retrieval_exemplars(cfg, index, smi, K_SHOT)
        prompt = build_prompt(cfg, smi, exemplars)
        probs = score_labels(model, tokenizer, prompt, cfg.label_words)
        probs_all.append(probs)
        y_true.append(y)
        if (i + 1) % 50 == 0:
            print(f"    [{dataset_name}/{regime}] {i+1}/{len(te_df)}  ({(i+1)/(time.time()-t0):.1f} mol/s)")

    probs_all = np.stack(probs_all)
    y_true = np.array(y_true)
    elapsed = time.time() - t0
    print(f"    done in {elapsed:.1f}s ({len(te_df)/elapsed:.2f} mol/s)")

    if cfg.task == "classification":
        pos_idx = cfg.label_words.index(cfg.positive_word)
        p_pos = probs_all[:, pos_idx]
        pred  = (p_pos >= 0.5).astype(int)
        auc = float(roc_auc_score(y_true, p_pos)) if len(set(y_true)) == 2 else float("nan")
        f1v = float(f1_score(y_true, pred)) if len(set(y_true)) == 2 else float("nan")
        return {
            "model": model_name, "dataset": dataset_name,
            "task": cfg.task, "regime": regime,
            "metric_primary": "roc_auc",   "value_primary":   round(auc, 4),
            "metric_secondary": "f1",      "value_secondary": round(f1v, 4),
            "n_train": n_train, "n_test": n_test_rows,
        }
    else:
        preds_cont = (probs_all * ESOL_MIDPOINTS[None, :]).sum(axis=1)
        y_cont = y_true.astype(np.float32)
        rmse = float(np.sqrt(mean_squared_error(y_cont, preds_cont)))
        mae  = float(mean_absolute_error(y_cont, preds_cont))
        return {
            "model": model_name, "dataset": dataset_name,
            "task": cfg.task, "regime": regime,
            "metric_primary": "rmse",   "value_primary":   round(rmse, 4),
            "metric_secondary": "mae",  "value_secondary": round(mae, 4),
            "n_train": n_train, "n_test": n_test_rows,
        }

def run_model_full_grid(model, tokenizer, model_name):
    rows = []
    for ds_name in ["BBBP", "BACE", "ESOL"]:
        tr_df, te_df = splits[ds_name]
        for regime in ["zero", "few", "retrieval"]:
            print(f"\n>>> {model_name} | {ds_name} | {regime}")
            rows.append(evaluate(model, tokenizer, model_name, ds_name, regime, len(tr_df), len(te_df)))
    return rows

## 6. Append-to-scoreboard helper

In [ ]:
RESULT_COLS = ["model", "dataset", "task", "regime",
               "metric_primary", "value_primary",
               "metric_secondary", "value_secondary",
               "n_train", "n_test"]

def append_to_scoreboard(new_rows):
    new_df = pd.DataFrame(new_rows)
    if os.path.exists(RESULTS_CSV):
        existing = pd.read_csv(RESULTS_CSV)
        if "regime" not in existing.columns:
            existing["regime"] = ""
        existing = existing[~existing[["model", "regime"]].apply(tuple, axis=1).isin(
            new_df[["model", "regime"]].apply(tuple, axis=1)
        )]
        combined = pd.concat([existing, new_df], ignore_index=True)
    else:
        combined = new_df
    combined = combined[RESULT_COLS]
    combined.to_csv(RESULTS_CSV, index=False)
    print(f"Wrote {len(new_df)} new rows. Total scoreboard: {len(combined)} rows.")

## 7. (Optional) Hugging Face login

Phi-4-mini-instruct is **sometimes gated**. If the model load below errors with `401 Client Error: Unauthorized`, uncomment and run the cell, paste an HF token (https://huggingface.co/settings/tokens, read scope), then re-run section 8.

In [ ]:
# from huggingface_hub import login
# login()  # interactive token prompt

## 8. Load Phi-4-mini-instruct (4-bit quantised)

~3.8B params. We load in 4-bit (`nf4` + double quant) so the full grid — including long retrieval prompts — fits comfortably on a T4 with headroom for KV-cache.

If you hit the `flash-attn` warning it's safe to ignore. Phi-4-mini ships with `trust_remote_code=False` by default in recent transformers; if your transformers version complains, add `trust_remote_code=True` to the load call.

In [ ]:
MODEL_ID_PHI4 = "microsoft/Phi-4-mini-instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {MODEL_ID_PHI4} in 4-bit...")
tok_phi4 = AutoTokenizer.from_pretrained(MODEL_ID_PHI4)
if tok_phi4.pad_token is None:
    tok_phi4.pad_token = tok_phi4.eos_token
mdl_phi4 = AutoModelForCausalLM.from_pretrained(
    MODEL_ID_PHI4,
    quantization_config=bnb_config,
    device_map="auto",
).eval()
print("Loaded Phi-4-mini | footprint:", f"{mdl_phi4.get_memory_footprint()/1024**2:.0f} MiB")

## 9. (Diagnostic) Confirm “yes” / “no” tokenise cleanly

Same check we ran for SmolLM3. If `yes`/`no` (with or without leading space) split into multiple tokens, `score_labels()` would read the wrong logit and the classification numbers would be unreliable. One quick cell, then move on.

In [ ]:
for w in ["yes", "no", " yes", " no", "Yes", "No"]:
    ids = tok_phi4(w, add_special_tokens=False).input_ids
    decoded = [tok_phi4.decode([i]) for i in ids]
    print(f"  {w!r:8s} -> ids={ids} | tokens={decoded}")

## 10. Run the full 9-cell grid

In [ ]:
phi4_rows = run_model_full_grid(mdl_phi4, tok_phi4, model_name="Phi-4-mini-prompt")
print("\nPhi-4-mini results:")
print(pd.DataFrame(phi4_rows).to_string(index=False))

append_to_scoreboard(phi4_rows)

del mdl_phi4, tok_phi4
gc.collect(); torch.cuda.empty_cache()

## 11. Final scoreboard view

In [ ]:
final = pd.read_csv(RESULTS_CSV)
print(f"Scoreboard rows: {len(final)}\n")
print(final.to_string(index=False))

## 12. Done

**Added in this notebook:** 9 rows — Phi-4-mini × 3 datasets × 3 regimes.

**Phase 4 — fully complete:**
- 04a: infrastructure + smoke test
- 04b: GPT-2 + SmolLM3-3B grids (18 rows)
- 04c: Phi-4-mini grid (9 rows)
- **Scoreboard total: 6 (Phase 3) + 27 (Phase 4) = 33 rows**

**What to look for once 04c finishes:**
- Is Phi-4-mini’s retrieval row consistently better than its zero-shot row? (Tests the “retrieval helps” hypothesis.)
- Is Phi-4-mini better than SmolLM3-3B on any dataset? (Tests “training-data curation beats raw size”.)
- Does any prompting setup beat the Phase 3 RF baseline? (Probably not — and that gap is the motivation for Phase 5.)

**Next (Phase 5):** LoRA fine-tune of the strongest prompting model on each dataset. The target to beat is now the full 33-row scoreboard.